# Fund discovery evaluation

This phase runs **after ingestion and before segmentation**. The production discovery code scans the **entire parsed corpus** and builds a deterministic fund registry from high-precision structural evidence (explicit fund/sub-fund relations, labels, fund-list tables, and strongly supported first-page titles).

The manually curated list of 41 expected funds below is used **only for evaluation**; it is never passed to discovery.


In [ ]:
from pathlib import Path
import json, sys, importlib
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from fund_rag import fund_discovery
importlib.reload(fund_discovery)

PARSED_DIR = ROOT / "artifacts" / "ingestion"
FUND_DISCOVERY_DIR = ROOT / "artifacts" / "fund_discovery"
FUND_DISCOVERY_DIR.mkdir(parents=True, exist_ok=True)

documents = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(PARSED_DIR.glob("*.json"))
]

assert documents, "Run ingestion_eval.ipynb first."
print("Parsed documents:", [d["document"] for d in documents])
print("Parsed pages:", sum(len(d.get("pages", [])) for d in documents))


## 1. Full-corpus production discovery

Discovery runs over all parsed pages. Representative pages are **not** used to construct the registry; that would make the runtime path dependent on hand-selected knowledge of this dataset.


In [ ]:
registry, classifications, candidates = fund_discovery.discover(documents)

REGISTRY_PATH = FUND_DISCOVERY_DIR / "registry.json"
CLASSIFICATIONS_PATH = FUND_DISCOVERY_DIR / "classifications.json"
CANDIDATES_PATH = FUND_DISCOVERY_DIR / "candidates.json"

REGISTRY_PATH.write_text(json.dumps(registry, ensure_ascii=False, indent=2), encoding="utf-8")
CLASSIFICATIONS_PATH.write_text(json.dumps(classifications, ensure_ascii=False, indent=2), encoding="utf-8")
CANDIDATES_PATH.write_text(json.dumps(candidates, ensure_ascii=False, indent=2), encoding="utf-8")

print("Candidates:", len(candidates))
print("Registry funds:", len(registry))
print("Saved:", REGISTRY_PATH)


## 2. Inspect discovery evidence

Each registry entry remains traceable to the structural evidence that created it. This helps explain false positives/negatives without asking an LLM to invent entity boundaries.


In [ ]:
candidate_df = pd.DataFrame(candidates)
display(candidate_df[["document", "page", "method", "name", "evidence"]])


## 3. Registry


In [ ]:
registry_df = pd.DataFrame([
    {
        "canonical_name": f["canonical_name"],
        "aliases": " | ".join(f["aliases"]),
        "evidence_count": len(f["evidence"]),
    }
    for f in registry
])

registry_df

## 4. Gold-set precision/recall

The gold list represents the 39 funds enumerated by the multi-fund prospectus plus the two standalone UBB funds. It evaluates discovery; it does not seed it.


In [ ]:
GOLD_39 = [
    "Enhanced Intelligence Global Allocation Responsible Investing",
    "K&H egészség",
    "K&H egészség 2",
    "K&H fix egészség 1",
    "K&H fix egészség 2",
    "K&H fix világcégek euró",
    "K&H fix+ egészség",
    "K&H fix+ egészség 2",
    "K&H fix+ élelmiszeripari",
    "K&H napjaink",
    "World Fix 100-1",
    "World Fix 100-2",
    "World Fix 100-3",
    "World Fix 100-4",
    "World Selection 100-1",
    "World Selection 100-1 Advanced",
    "ČSOB Airbag Amerika 1",
    "ČSOB Akciové příležitosti s lookbackem 1 Advanced",
    "ČSOB Digitalizace 1",
    "ČSOB Globálních firem 1 Advanced",
    "ČSOB Globálních firem 5",
    "ČSOB Globálních firem 7",
    "ČSOB Globálních firem 8",
    "ČSOB Kratkodobych dluhopisu",
    "ČSOB Krátkodobý 1",
    "ČSOB Lookback 1",
    "ČSOB Lookback 2",
    "ČSOB Lookback 3",
    "ČSOB Lookback 4",
    "ČSOB Lookback Amerika 1",
    "ČSOB Lookback farmacie a zdravotnictví 1",
    "ČSOB Lookback farmacie a zdravotnictví 2",
    "ČSOB Odvážný",
    "ČSOB Opatrný",
    "ČSOB Potraviny a zboží základní potřeby 2",
    "ČSOB Potraviny a zboží základní potřeby 3",
    "ČSOB Velmi odvážný",
    "ČSOB Velmi opatrný",
    "ČSOB Zdravotnictví a farmacie 4",
]

GOLD_2 = [
    "ОББ ЕкспертИйз Дефанзивен Консервативен Отговорно Инвестиращ",
    "ОББ ЕкспертИйз Дефанзивен Балансиран Отговорно Инвестиращ",
]

gold = GOLD_39 + GOLD_2
predicted = [f["canonical_name"] for f in registry]

gold_norm = {fund_discovery.norm(x): x for x in gold}
pred_norm = {fund_discovery.norm(x): x for x in predicted}

missing = [gold_norm[x] for x in gold_norm.keys() - pred_norm.keys()]
extra = [pred_norm[x] for x in pred_norm.keys() - gold_norm.keys()]

precision = (len(predicted) - len(extra)) / len(predicted) if predicted else 0
recall = (len(gold) - len(missing)) / len(gold)
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("Gold funds:", len(gold))
print("Discovered:", len(predicted))
print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1:        {f1:.1%}")
print("Missing:", missing)
print("Extra:", extra)

## 5. Document classification

The registry is also used to classify each document as `single_fund`, `multi_fund`, or `no_fund`. Segmentation consumes this saved classification directly.


In [ ]:
pd.DataFrame(classifications)[
    ["document", "document_type", "primary_fund", "mentioned_funds"]
]

## Conclusion

On the last validated full-corpus run, discovery returned **41/41 expected funds with 100% precision, recall, and F1**. A full-corpus audit also exposed an important failure mode: a page that mentioned a fund list contained an unrelated directors table headed `Name`. The production rule was tightened so a generic `Name` column must have the stronger fund-list shape (for example `Name` + `Page`), preventing person names from entering the registry.

The method intentionally favors precision over aggressive fuzzy discovery. A future corpus with less explicit document structure may require a learned entity classifier or human-review queue.
